In [16]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [17]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

train_df.shape

(2000, 8)

In [18]:
import torch
import random
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEBERTA_MODEL = "microsoft/deberta-v3-small"

deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL)

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_MODEL,
    num_labels=2
)

deberta_model.eval()

ROBERTA_MODEL = "roberta-base"

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_MODEL,
    num_labels=2
)

roberta_model.eval()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias         

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [19]:
import torch

row = train_df.iloc[25]

prompt = row["prompt"]

options = {
    "A": row["A"],
    "B": row["B"],
    "C": row["C"],
    "D": row["D"],
    "E": row["E"],
}

scores = {}

for option, answer in options.items():

    inputs = deberta_tokenizer(
        prompt,
        answer,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)

    # Probability of the positive class
    scores[option] = probs[0][1].item()

print(scores)

{'A': 0.513671875, 'B': 0.5048828125, 'C': 0.51220703125, 'D': 0.51025390625, 'E': 0.5263671875}


In [20]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "roberta-base"

roberta_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

roberta_model.eval()

roberta_scores = {}

for option, answer in options.items():

    inputs = roberta_tokenizer(
        prompt,
        answer,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = roberta_model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)

    roberta_scores[option] = probs[0][1].item()

print(roberta_scores)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'A': 0.4952384829521179, 'B': 0.4963659346103668, 'C': 0.49683213233947754, 'D': 0.4938080608844757, 'E': 0.4986579120159149}


In [21]:
ensemble_scores = {}

for option in options.keys():
    ensemble_scores[option] = (
        scores[option] + roberta_scores[option]
    ) / 2

print(ensemble_scores)

{'A': 0.504455178976059, 'B': 0.5006243735551834, 'C': 0.5045195817947388, 'D': 0.5020309835672379, 'E': 0.5125125497579575}


In [22]:
weighted_scores = {}

for option in options.keys():
    weighted_scores[option] = (
        0.7 * scores[option] +
        0.3 * roberta_scores[option]
    )

print(weighted_scores)

best_option = max(weighted_scores, key=weighted_scores.get)

print(best_option)

{'A': 0.5081418573856353, 'B': 0.50232774913311, 'C': 0.5075945615768432, 'D': 0.5053201526403427, 'E': 0.5180544048547744}
E


In [23]:
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(len(test_df))

500


In [24]:
tta_changes = 0

instruction = "Answer the following multiple-choice question carefully: "

for idx in range(50):

    row = test_df.iloc[idx]

    original_prompt = row["prompt"]
    augmented_prompt = instruction + row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"],
    }

    original_scores = {}
    augmented_scores = {}

    #Original prompt
    for option, answer in options.items():

        inputs = deberta_tokenizer(
            original_prompt,
            answer,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = deberta_model(**inputs)

        probs = torch.softmax(outputs.logits, dim=-1)
        original_scores[option] = probs[0][1].item()

    #Augmented prompt
    for option, answer in options.items():

        inputs = deberta_tokenizer(
            augmented_prompt,
            answer,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = deberta_model(**inputs)

        probs = torch.softmax(outputs.logits, dim=-1)
        augmented_scores[option] = probs[0][1].item()

    # Top-1 without TTA
    original_top = max(original_scores, key=original_scores.get)

    # Average probabilities
    tta_scores = {
        option: (original_scores[option] + augmented_scores[option]) / 2
        for option in options
    }

    # Top-1 with TTA
    tta_top = max(tta_scores, key=tta_scores.get)

    if original_top != tta_top:
        tta_changes += 1

print("Number of changed predictions:", tta_changes)

Number of changed predictions: 17


In [25]:
different_predictions = 0

for idx in range(100):

    row = test_df.iloc[idx]

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"],
    }

    deberta_scores = {}
    roberta_scores = {}

    # DeBERTa inference
    for option, answer in options.items():

        inputs = deberta_tokenizer(
            prompt,
            answer,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = deberta_model(**inputs)

        probs = torch.softmax(outputs.logits, dim=-1)
        deberta_scores[option] = probs[0][1].item()

    # RoBERTa inference
    for option, answer in options.items():

        inputs = roberta_tokenizer(
            prompt,
            answer,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = roberta_model(**inputs)

        probs = torch.softmax(outputs.logits, dim=-1)
        roberta_scores[option] = probs[0][1].item()

    # Top-1 DeBERTa
    deberta_top = max(deberta_scores, key=deberta_scores.get)

    # Weighted Ensemble
    weighted_scores = {
        option: 0.7 * deberta_scores[option] + 0.3 * roberta_scores[option]
        for option in options
    }

    ensemble_top = max(weighted_scores, key=weighted_scores.get)

    if deberta_top != ensemble_top:
        different_predictions += 1

print("Different Top-1 predictions:", different_predictions)

Different Top-1 predictions: 1


In [26]:
positive_confidence_gain = 0

for idx in range(100):

    row = test_df.iloc[idx]

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"],
    }

    deberta_scores = {}
    roberta_scores = {}

    # DeBERTa
    for option, answer in options.items():

        inputs = deberta_tokenizer(
            prompt,
            answer,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = deberta_model(**inputs)

        probs = torch.softmax(outputs.logits, dim=-1)
        deberta_scores[option] = probs[0][1].item()

    # RoBERTa
    for option, answer in options.items():

        inputs = roberta_tokenizer(
            prompt,
            answer,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = roberta_model(**inputs)

        probs = torch.softmax(outputs.logits, dim=-1)
        roberta_scores[option] = probs[0][1].item()

    # Weighted Ensemble
    weighted_scores = {
        option: 0.7 * deberta_scores[option] + 0.3 * roberta_scores[option]
        for option in options
    }

    deberta_confidence = max(deberta_scores.values())
    ensemble_confidence = max(weighted_scores.values())

    confidence_gain = ensemble_confidence - deberta_confidence

    if confidence_gain > 0:
        positive_confidence_gain += 1

print("Positive confidence gain:", positive_confidence_gain)

Positive confidence gain: 37


In [27]:
top3_changes = 0

for idx in range(100):

    row = test_df.iloc[idx]

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"],
    }

    deberta_scores = {}
    roberta_scores = {}

    # DeBERTa
    for option, answer in options.items():

        inputs = deberta_tokenizer(
            prompt,
            answer,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = deberta_model(**inputs)

        probs = torch.softmax(outputs.logits, dim=-1)
        deberta_scores[option] = probs[0][1].item()

    # RoBERTa
    for option, answer in options.items():

        inputs = roberta_tokenizer(
            prompt,
            answer,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = roberta_model(**inputs)

        probs = torch.softmax(outputs.logits, dim=-1)
        roberta_scores[option] = probs[0][1].item()

    # Weighted Ensemble
    weighted_scores = {
        option: 0.7 * deberta_scores[option] + 0.3 * roberta_scores[option]
        for option in options
    }

    # Ordered Top-3
    deberta_top3 = sorted(deberta_scores, key=deberta_scores.get, reverse=True)[:3]
    ensemble_top3 = sorted(weighted_scores, key=weighted_scores.get, reverse=True)[:3]

    if deberta_top3 != ensemble_top3:
        top3_changes += 1

print("Rows with Top-3 changes:", top3_changes)

Rows with Top-3 changes: 7


In [28]:
def apk(actual, predicted, k=3):
    """Average Precision at k for one sample."""
    predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i, p in enumerate(predicted):
        if p == actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)

    return score


predictions = []
actuals = []

# Use the first 100 validation samples
for idx in range(100):

    row = train_df.iloc[idx]

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"],
    }

    deberta_scores = {}
    roberta_scores = {}

    # DeBERTa
    for option, answer in options.items():

        inputs = deberta_tokenizer(
            prompt,
            answer,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = deberta_model(**inputs)

        probs = torch.softmax(outputs.logits, dim=-1)
        deberta_scores[option] = probs[0][1].item()

    # RoBERTa
    for option, answer in options.items():

        inputs = roberta_tokenizer(
            prompt,
            answer,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = roberta_model(**inputs)

        probs = torch.softmax(outputs.logits, dim=-1)
        roberta_scores[option] = probs[0][1].item()

    # Weighted ensemble
    weighted_scores = {
        option: 0.7 * deberta_scores[option] + 0.3 * roberta_scores[option]
        for option in options
    }

    top3 = sorted(weighted_scores, key=weighted_scores.get, reverse=True)[:3]

    predictions.append(top3)
    actuals.append(row["answer"])

# MAP@3
map3 = sum(apk(a, p, 3) for a, p in zip(actuals, predictions)) / len(actuals)

print(f"MAP@3: {map3:.4f}")

MAP@3: 0.4200
